This notebooks point is to work as a place to freely explore the datasets

Tarkastellaan Natura 2000 dataa

In [2]:
from pathlib import Path

print(Path.cwd())

/Users/merihilden/code/projekti/parcel-nature-screening/notebooks


In [3]:
import geopandas as gpd

gdf = gpd.read_file("../data/raw/natura/natura2000sac_alueet.shp")

print(gdf.crs)
print(gdf.columns.tolist())

gdf.head()

EPSG:3067
['objectid', 'naturatunn', 'suojeluper', 'versiotunn', 'nimisuomi', 'nimiruotsi', 'aluetyyppi', 'paatospvm', 'paatospala', 'paatospitu', 'meripalapr', 'paatosasia', 'ensisijlaj', 'aluejaviiv', 'vpdsuoj', 'lisatieto', 'luontipvm', 'muutospvm', 'paattymisp', 'area_m2', 'perimeter_', 'tietolomak', 'datablanke', 'tiivistelm', 'sammanfatt', 'geometry']


,objectid,naturatunn,suojeluper,versiotunn,nimisuomi,nimiruotsi,aluetyyppi,paatospvm,paatospala,paatospitu,...,luontipvm,muutospvm,paattymisp,area_m2,perimeter_,tietolomak,datablanke,tiivistelm,sammanfatt,geometry
0,4,FI0100005,SACFI0100005,20160913000000,Tammisaaren ja Hangon saariston ja Pohjanpitäj...,De skyddsvärda marina områdena i Ekenäs och Ha...,SAC/SPA,2015-03-27,52630.0,0.0,...,2016-09-13,NaT,NaT,5.257942e+08,988805.2414,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/2018/ti...,"MULTIPOLYGON (((300311.836 6650178.304, 300376..."
1,525,FI0425005,SACFI0425005,20160913000000,Pappilansaari - Lupinlahti,Pappilansaari-Lupinlahti,SAC,2015-03-27,400.0,0.0,...,2016-09-13,NaT,NaT,3.995545e+06,20579.2388,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/EiRuots...,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/EiRuots...,"POLYGON ((511553.08 6713857.315, 511654.039 67..."
2,1390,FI1200608,SACFI1200608,20160913000000,Räätäkangas,Räätäkangas,SAC,2015-03-27,667.0,0.0,...,2016-09-13,NaT,NaT,6.660675e+06,19056.4922,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/EiRuots...,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/EiRuots...,"POLYGON ((585786.313 7099744.045, 585820.799 7..."
3,535,FI0500001,SACFI0500001,20160913000000,Kolovesi - Vaaluvirta - Pyttyselkä,Kolovesi - Vaaluvirta - Pyttyselkä,SAC,2015-03-27,7986.0,0.0,...,2016-09-13,NaT,NaT,7.979903e+07,96243.9426,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/EiRuots...,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/EiRuots...,"POLYGON ((590995.441 6905779.511, 590872.445 6..."
4,150,FI0200072,SACFI0200072,20160913000000,Uudenkaupungin saaristo,Nystads skärgård,SAC/SPA,2015-03-27,56992.0,0.0,...,2016-09-13,NaT,NaT,5.694506e+08,222625.0579,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/EiRuots...,http://paikkatieto.ymparisto.fi/natura/2018/ti...,http://paikkatieto.ymparisto.fi/natura/EiRuots...,"MULTIPOLYGON (((182592.248 6761513.775, 182592..."


Alla oleva etsii kaikki layerit automaattisesti, kun sille antaa wfs urlin

In [2]:
from __future__ import annotations

import xml.etree.ElementTree as ET
from dataclasses import dataclass

import requests


@dataclass
class WFSLayer:
    name: str
    title: str | None = None
    abstract: str | None = None


def get_wfs_capabilities_xml(wfs_url: str) -> str:
    params = {
        "service": "WFS",
        "request": "GetCapabilities",
    }

    response = requests.get(wfs_url, params=params, timeout=60)
    response.raise_for_status()

    return response.text


def list_wfs_layers(wfs_url: str) -> list[WFSLayer]:
    xml_text = get_wfs_capabilities_xml(wfs_url)

    root = ET.fromstring(xml_text)

    namespaces = {
        "wfs": "http://www.opengis.net/wfs/2.0",
        "ows": "http://www.opengis.net/ows/1.1",
    }

    layers: list[WFSLayer] = []

    for feature_type in root.findall(".//wfs:FeatureType", namespaces):
        name_el = feature_type.find("wfs:Name", namespaces)
        title_el = feature_type.find("wfs:Title", namespaces)
        abstract_el = feature_type.find("wfs:Abstract", namespaces)

        if name_el is None or not name_el.text:
            continue

        layers.append(
            WFSLayer(
                name=name_el.text.strip(),
                title=title_el.text.strip() if title_el is not None and title_el.text else None,
                abstract=abstract_el.text.strip()
                if abstract_el is not None and abstract_el.text
                else None,
            )
        )

    return layers


def print_wfs_layers(wfs_url: str) -> None:
    layers = list_wfs_layers(wfs_url)

    print(f"Found {len(layers)} layers:\n")

    for layer in layers:
        print(f"Name:  {layer.name}")
        print(f"Title: {layer.title}")
        print("-" * 80)

Tarkastellaan metsäkeskuksen dataa

In [3]:
xml = get_wfs_capabilities_xml(
    "https://avoin.metsakeskus.fi/rajapinnat/v1/habitat/ows"
)

print(xml[:3000])

<?xml version="1.0" encoding="UTF-8"?><wfs:WFS_Capabilities version="2.0.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://www.opengis.net/wfs/2.0" xmlns:wfs="http://www.opengis.net/wfs/2.0" xmlns:ows="http://www.opengis.net/ows/1.1" xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:fes="http://www.opengis.net/fes/2.0" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xs="http://www.w3.org/2001/XMLSchema" xsi:schemaLocation="http://www.opengis.net/wfs/2.0 https://avoin.metsakeskus.fi/rajapinnat/schemas/wfs/2.0/wfs.xsd http://inspire.ec.europa.eu/schemas/inspire_dls/1.0 https://inspire.ec.europa.eu/schemas/inspire_dls/1.0/inspire_dls.xsd" xmlns:xml="http://www.w3.org/XML/1998/namespace" xmlns:inspire_dls="http://inspire.ec.europa.eu/schemas/inspire_dls/1.0" xmlns:inspire_common="http://inspire.ec.europa.eu/schemas/common/1.0" xmlns:v1="https://avoin.metsakeskus.fi/rajapinnat/v1" updateSequence="13273"><ows:ServiceIdentification><ows:Title>Avoimen metsätiedon WFS-palve

In [15]:
import geopandas as gpd


url = (
    "https://avoin.metsakeskus.fi/rajapinnat/v1/stand/wfs"
    "?service=WFS"
    "&version=2.0.0"
    "&request=GetFeature"
    f"&typeNames={"v1:stand"}"
    "&outputFormat=application/json"
    "&count=10"
)

gdf = gpd.read_file(url)
gdf.head()

,id,STANDNUMBER,STANDNUMBEREXTENSION,MAINGROUP,SUBGROUP,FERTILITYCLASS,SOILTYPE,DRAINAGESTATE,DITCHINGYEAR,DEVELOPMENTCLASS,...,PULPWOODVOLUME,VOLUMEGROWTH,TREESTANDDATE,CUTTINGTYPE,CUTTINGPROPOSALYEAR,SILVICULTURETYPE,SILVICULTUREPROPOSALYEAR,CREATIONTIME,UPDATETIME,geometry
0,stand.39524392,8,None,1,1,3,10,1,None,03,...,108.24,7.86,2026-01-01 00:00:00+02:00,3.0,2028.0,NaN,NaN,2021-10-25 08:14:37+03:00,2025-09-01 16:32:05.707000+03:00,"POLYGON ((467046.109 6940476.049, 467042.602 6..."
1,stand.39524393,2,None,1,1,3,10,1,None,04,...,93.14,7.27,2026-01-01 00:00:00+02:00,5.0,2026.0,1.0,2026.0,2021-10-25 08:14:37+03:00,2025-09-01 16:32:05.707000+03:00,"POLYGON ((466847.407 6940522.78, 466846.402 69..."
2,stand.39524394,1,None,1,1,3,10,1,None,03,...,141.18,8.80,2026-01-01 00:00:00+02:00,3.0,2026.0,NaN,NaN,2021-10-25 08:14:38+03:00,2025-09-01 16:32:05.707000+03:00,"POLYGON ((466903.837 6940510.035, 466904.366 6..."
3,stand.39524395,3,None,1,1,2,20,1,None,03,...,150.50,10.38,2026-01-01 00:00:00+02:00,3.0,2026.0,NaN,NaN,2021-10-25 08:14:38+03:00,2025-09-01 16:32:05.707000+03:00,"POLYGON ((466962.575 6940620.631, 466962.046 6..."
4,stand.39524396,6,None,1,1,3,10,1,None,02,...,60.93,9.80,2026-01-01 00:00:00+02:00,2.0,2033.0,4.0,2026.0,2021-10-25 08:14:38+03:00,2025-09-01 16:32:05.707000+03:00,"POLYGON ((466960.678 6940337.674, 466964.624 6..."
